# Task 1 EDA: `handicaps` in Link-Lives Release 2 (1801, 1850, 1860, 1880, 1901)

**Scope.** This notebook describes what the data shows. It decides nothing: the Task 1 method, whether to use an LLM for categories, the embedding scope and the File B impact definition are the user's decisions after this EDA (see the closing section "Facts for open decisions").

**Data handling.** Only aggregates, distributions, plots and value-frequency tables are displayed (D-02). No record-level rows are shown: no `pa_id` or other identifier columns, no names next to other attributes, no raw-frame previews. All columns are stored as strings in the parquet files (D-06), so numeric readings below are explicit casts that report their parse failures.

**Reading rule.** An empty `handicaps` value is not evidence of "no handicap". It may mean the question was not asked or the value was not transcribed. Column descriptions below describe what the data shows, not official definitions; the Link-Lives Release 2 Guide is the authority.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display
from rapidfuzz import fuzz, process

# nbconvert runs the kernel in the notebook's directory, so scripts/ is one level up.
sys.path.insert(0, str(Path.cwd().resolve().parent / "scripts"))
import project_paths as pp

pl.Config.set_tbl_rows(60)
pl.Config.set_fmt_str_lengths(80)

YEARS = pp.CENSUS_YEARS
LF = {y: pl.scan_parquet(pp.census_parquet(y)) for y in YEARS}
MIN_EXAMPLE_COUNT = 5
CAPACITY_LINES = (200, 300, 360)
CLUSTER_THRESHOLDS = (80, 85, 90)
REF_THRESHOLD = 85
FACTS = {}


def filled(col):
    # Not null and not blank after strip.
    return pl.col(col).is_not_null() & (pl.col(col).str.strip_chars() != "")


def trivial_norm(expr):
    # lowercase, punctuation runs -> space, collapse whitespace, strip.
    return (
        expr.str.to_lowercase()
        .str.replace_all(r"[^\w\s]+", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def fact(key, value):
    if isinstance(value, float):
        value = round(value, 4)
    FACTS[key] = value
    print(f"FACT {key}={value}")


def safe_div(a, b):
    return a / b if b else 0.0

## 1. `handicaps` per census year

In [ ]:
def year_summary(y):
    row = (
        LF[y]
        .select(
            pl.len().alias("rows_n"),
            filled("handicaps").sum().alias("filled_n"),
            pl.col("handicaps").filter(filled("handicaps")).n_unique().alias("raw_distinct_n"),
        )
        .collect()
        .row(0, named=True)
    )
    row["year"] = y
    row["fill_rate"] = safe_div(row["filled_n"], row["rows_n"])
    return row


summary = pl.DataFrame([year_summary(y) for y in YEARS]).select(
    "year", "rows_n", "filled_n", "fill_rate", "raw_distinct_n"
)
for r in summary.iter_rows(named=True):
    fact(f"task1.rows.{r['year']}", r["rows_n"])
    fact(f"task1.fill_rate.{r['year']}", r["fill_rate"])
    fact(f"task1.raw_distinct.{r['year']}", r["raw_distinct_n"])
summary

**Caveat next to these numbers.** "Missing" here means null or blank after stripping whitespace. An empty `handicaps` value may mean the question was not asked or the value was not transcribed; it must not be read as "no handicap". `raw_distinct_n` counts distinct filled strings exactly as stored. Years with very few filled values are shown as they are, and all rates are zero-safe.

## 2. Column overview (what the data shows)

Storage is all-String by design (D-06), so "type" below is an **observed parse profile**, not an official type: `int_parse_share` is the share of filled values that parse as a whole number (an explicit integer cast, or a float cast with no fractional part, because numbers such as ages are stored like "34.0"), and a column is labelled "integer-like" when that share is at least 0.95, otherwise "text". `pct_missing` counts null or blank values (in percent). `n_unique` counts distinct filled values. `examples` lists up to three values that occur at least `MIN_EXAMPLE_COUNT` times, with their counts. Identifier columns, free-text columns and name columns are masked as "(not shown)".

These descriptions say what the columns contain in this data. Their meanings are **not** asserted here: the Release 2 Guide is the authority.

In [ ]:
MASK_EXAMPLE_COLUMNS = {
    "pa_id", "Indtastningsnr", "løbenr_i_indtastning", "Kildehenvisning",
    "Husstands_familienr", "Matr_nr_adresse", "Adresse", "Matrikel", "gadenr",
    "fødselsdato", "Kommentarer", "kildekommentar", "source_id",
    # name columns are masked too: names are never displayed
    "navn", "ejernavn", "ejendom_navn",
}


def whole_number(expr):
    # explicit casts on the stripped string; "34.0" counts as whole, "34.5" and "abc" do not
    f = expr.cast(pl.Float64, strict=False)
    return f.is_not_null() & f.is_finite() & (f == f.floor())


def column_profile(lf, col, rows_n, mask):
    vc = (
        lf.select(pl.col(col).str.strip_chars().alias("v"))
        .filter(pl.col("v").is_not_null() & (pl.col("v") != ""))
        .group_by("v")
        .len()
        .rename({"len": "n"})
        .collect()
    )
    filled_n = int(vc["n"].sum()) if vc.height else 0
    int_n = (
        int(vc.filter(whole_number(pl.col("v")))["n"].sum())
        if vc.height
        else 0
    )
    int_share = safe_div(int_n, filled_n)
    if mask:
        examples = "(not shown)"
    else:
        top = (
            vc.filter(pl.col("n") >= MIN_EXAMPLE_COUNT)
            .sort(["n", "v"], descending=[True, False])
            .head(3)
        )
        examples = "; ".join(f"{v} ({n})" for v, n in top.iter_rows())
    return {
        "column": col,
        "type_profile": "integer-like" if int_share >= 0.95 else "text",
        "int_parse_share": round(int_share, 4),
        "pct_missing": round(100 * (1 - safe_div(filled_n, rows_n)), 2),
        "n_unique": vc.height,
        "examples": examples,
    }


def column_overview(lf, year):
    rows_n = lf.select(pl.len()).collect().item()
    cols = lf.collect_schema().names()
    return pl.DataFrame(
        [column_profile(lf, c, rows_n, c in MASK_EXAMPLE_COLUMNS) for c in cols],
        schema={
            "column": pl.String, "type_profile": pl.String, "int_parse_share": pl.Float64,
            "pct_missing": pl.Float64, "n_unique": pl.UInt32, "examples": pl.String,
        },
    )


def alder_tal_findings(lf):
    # explicit cast of filled alder_tal: not numeric at all, or numeric with a fractional part
    vc = (
        lf.select(pl.col("alder_tal").str.strip_chars().alias("v"))
        .filter(pl.col("v").is_not_null() & (pl.col("v") != ""))
        .group_by("v").len().rename({"len": "n"}).collect()
    )
    if not vc.height:
        return 0, 0
    f = pl.col("v").cast(pl.Float64, strict=False)
    not_numeric = int(vc.filter(f.is_null() | f.is_nan() | f.is_infinite())["n"].sum())
    fractional = int(vc.filter(f.is_finite() & (f != f.floor()))["n"].sum())
    return not_numeric, fractional


OVERVIEW = {}
for y in YEARS:
    OVERVIEW[y] = column_overview(LF[y], y)
    bad, frac = alder_tal_findings(LF[y])
    fact(f"task1.alder_tal_parse_fail.{y}", bad)
    fact(f"task1.alder_tal_fractional.{y}", frac)

### Column overview 1801

In [ ]:
OVERVIEW[1801]

### Column overview 1850

In [ ]:
OVERVIEW[1850]

### Column overview 1860

In [ ]:
OVERVIEW[1860]

### Column overview 1880

In [ ]:
OVERVIEW[1880]

### Column overview 1901

In [ ]:
OVERVIEW[1901]

`task1.alder_tal_parse_fail.<year>` above counts filled `alder_tal` values that do not parse as a number at all, and `task1.alder_tal_fractional.<year>` those that parse but have a fractional part. Both are findings about the data, not corrected. Note that `alder_tal` is stored in a float format (for example "34.0"), so a plain integer cast would reject nearly every value; the explicit cast therefore goes through a float and then checks for a whole number.

## 3. Missingness by geography

The geographic unit is the **parish key** `Amt|Herred|Sogn`, built as a composite because parish names repeat across counties and districts. For each year, a unit is **empty** when none of its records has a filled `handicaps` value. Records with no geography at all are left out of the unit counts and reported separately as `no_geo_rows_n`.

In [ ]:
GEO = ["Amt", "Herred", "Sogn"]


def geo_key():
    parts = [pl.when(filled(c)).then(pl.col(c).str.strip_chars()) for c in GEO]
    return pl.concat_str(parts, separator="|", ignore_nulls=True)


def parish_summary(y):
    per_unit = (
        LF[y]
        .select(geo_key().alias("key"), filled("handicaps").alias("h"))
        .filter(pl.col("key") != "")
        .group_by("key")
        .agg(pl.col("h").sum().alias("filled_n"))
        .collect()
    )
    units_n = per_unit.height
    empty_n = int(per_unit.filter(pl.col("filled_n") == 0).height)
    no_geo = (
        LF[y].select((geo_key() == "").sum().alias("n")).collect().item()
    )
    return {
        "year": y,
        "units_n": units_n,
        "empty_units_n": empty_n,
        "empty_unit_share": safe_div(empty_n, units_n),
        "no_geo_rows_n": int(no_geo),
    }


parish_table = pl.DataFrame([parish_summary(y) for y in YEARS])
for r in parish_table.iter_rows(named=True):
    fact(f"task1.empty_parish_share.{r['year']}", r["empty_unit_share"])
parish_table

**Caveat next to these numbers.** A parish with no filled values suggests "not asked / not transcribed" rather than "no handicap". 1801 is the clearest case: almost every parish is empty, which points at the source or the transcription, not at an absence of handicaps. A parish with a filled share is not evidence of completeness either.

In [ ]:
def fill_by_amt(y):
    return (
        LF[y]
        .select(
            pl.when(filled("Amt")).then(pl.col("Amt").str.strip_chars()).otherwise(pl.lit("(missing)")).alias("Amt"),
            filled("handicaps").alias("h"),
        )
        .group_by("Amt")
        .agg((pl.col("h").sum() / pl.len()).alias(str(y)))
        .collect()
    )


amt_pivot = fill_by_amt(YEARS[0])
for y in YEARS[1:]:
    amt_pivot = amt_pivot.join(fill_by_amt(y), on="Amt", how="full", coalesce=True)
amt_pivot = amt_pivot.sort("Amt").with_columns(pl.col(pl.Float64).round(4))
amt_pivot

Fill rate by county (`Amt`) across years; a blank cell means the county does not occur in that census. A county whose rate is zero or close to it in one year but not in others again points at "not asked / not transcribed".

## Facts for open decisions

Every `FACT` line below was computed in this notebook. Later sections are added above this one.

In [ ]:
for k, v in FACTS.items():
    print(f"FACT {k}={v}")